In [1]:

from support.search import validate_search

# Entradas válidas
validate_search("wireless headphones", 5)
validate_search("bluetooth speaker", 1)
validate_search("gaming mouse", 20)

print("Valid inputs passed")


Valid inputs passed


In [2]:

invalid_cases = [
    ("", 5),
    ("   ", 5),
    ("headphones", 0),
    ("headphones", 21),
    ("headphones", 2.5),
    ("headphones", True),
]

for query, top_k in invalid_cases:
    try:
        validate_search(query, top_k)
        print(f"FAILED: {query!r}, {top_k!r}")
    except ValueError as error:
        print(f"PASSED: {error}")


PASSED: Query cannot be empty
PASSED: Query cannot be empty
PASSED: top_k must be an integer between 1 and 20
PASSED: top_k must be an integer between 1 and 20
PASSED: top_k must be an integer between 1 and 20
PASSED: top_k must be an integer between 1 and 20


## 2 Similitud de Cocenos Y top K 

In [3]:

import torch

product_vectors = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [0.8, 0.6],
])

query_vector = torch.tensor([1.0, 0.0])

similarities = product_vectors @ query_vector

print("Shape:", similarities.shape)
print("Similarities:", similarities)


Shape: torch.Size([3])
Similarities: tensor([1.0000, 0.0000, 0.8000])


In [4]:

top_k = 2

top_values, top_indices = torch.topk(
    similarities,
    k=top_k,
    largest=True,
    sorted=True,
)

print("Top values:", top_values)
print("Top indices:", top_indices)


Top values: tensor([1.0000, 0.8000])
Top indices: tensor([0, 2])


In [8]:
import importlib
import support.search

importlib.reload(support.search)

search_top_k = support.search.search_top_k

In [9]:
from support.search import search_top_k

values, indices = search_top_k(
    product_vectors,
    query_vector,
    top_k=2,
)

print("Values:", values)
print("Indices:", indices)

assert values.tolist() == [1.0, 0.800000011920929]
assert indices.tolist() == [0, 2]

print("Top-K test passed")
  

Values: tensor([1.0000, 0.8000])
Indices: tensor([0, 2])
Top-K test passed


In [10]:

values, indices = search_top_k(
    product_vectors,
    query_vector,
    top_k=5,
)

print("Values:", values)
print("Indices:", indices)
print("Number of results:", len(values))

assert len(values) == 3
assert indices.tolist() == [0, 2, 1]

print("Top-K limit test passed")


Values: tensor([1.0000, 0.8000, 0.0000])
Indices: tensor([0, 2, 1])
Number of results: 3
Top-K limit test passed


In [11]:

example_ids = ["P001", "P002", "P003"]

for index in indices.tolist():
    print("Index:", index, "Product ID:", example_ids[index])


Index: 0 Product ID: P001
Index: 2 Product ID: P003
Index: 1 Product ID: P002


In [12]:

example_products = {
    "P001": {"title": "Wireless Headphones"},
    "P002": {"title": "HDMI Cable"},
    "P003": {"title": "Bluetooth Speaker"},
}

example_review_counts = {
    "P001": 12,
    "P002": 8,
    "P003": 15,
}

product_id = example_ids[0]

print("Product ID:", product_id)
print("Title:", example_products[product_id]["title"])
print("Reviews:", example_review_counts[product_id])


Product ID: P001
Title: Wireless Headphones
Reviews: 12


In [13]:
import importlib
import json
import support.search

importlib.reload(support.search)

example_products = {
    product_id: {
        "product_id": product_id,
        "title": title,
        "features": [],
        "description": [],
        "categories": [],
        "family": "Electronics",
    }
    for product_id, title in [
        ("P001", "Wireless Headphones"),
        ("P002", "HDMI Cable"),
        ("P003", "Bluetooth Speaker"),
    ]
}

values, indices = support.search.search_top_k(
    product_vectors,
    query_vector,
    top_k=2,
)

results = support.search.build_search_results(
    values,
    indices,
    example_ids,
    example_products,
    example_review_counts,
)

print(json.dumps(results, indent=2))   

[
  {
    "product_id": "P001",
    "title": "Wireless Headphones",
    "similarity": 1.0,
    "review_count": 12
  },
  {
    "product_id": "P003",
    "title": "Bluetooth Speaker",
    "similarity": 0.800000011920929,
    "review_count": 15
  }
]


## 3. Pruebas con datos reales

In [14]:
from pathlib import Path

from support.data import load, catalog_text, prepare_review_texts
from support.settings import load_settings

project_dir = Path.cwd()

settings = load_settings(project_dir / "config.json")
reviews, products = load(project_dir / "datasets")

product_ids = list(products.keys())

print("Products:", len(products))
print("Reviews:", len(reviews))
print("Product IDs:", len(product_ids))
print("Model:", settings.model_name)    

Products: 600
Reviews: 6992
Product IDs: 600
Model: sentence-transformers/all-MiniLM-L6-v2


In [15]:
from support.encoder import (
    load_model,
    encode_texts,
    summarize_reviews,
    combine_embeddings,
)

catalog_texts = [
    catalog_text(products[product_id])
    for product_id in product_ids
]

review_texts, review_ids, review_product_ids = prepare_review_texts(reviews)

model = load_model(settings)

print("Catalog texts:", len(catalog_texts))
print("Review texts:", len(review_texts))
print("Model loaded successfully")

c:\Users\Gabri\Documents\GitHub\Programacion_Para_IA\programacion_ai\amazon-products\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10641.77it/s]


Catalog texts: 600
Review texts: 6992
Model loaded successfully


In [16]:
catalog_embeddings = encode_texts(
    model,
    catalog_texts,
    settings.batch_size,
)

review_embeddings = encode_texts(
    model,
    review_texts,
    settings.batch_size,
)

review_summaries = summarize_reviews(
    product_ids,
    review_product_ids,
    review_embeddings,
)

product_embeddings = combine_embeddings(
    catalog_embeddings,
    review_summaries,
)

print("Product embeddings:", product_embeddings.shape)

Product embeddings: torch.Size([600, 384])


In [17]:
norms = torch.linalg.vector_norm(product_embeddings, dim=1)

print("Product IDs:", len(product_ids))
print("Matrix rows:", product_embeddings.shape[0])
print("First 5 norms:", norms[:5])

assert len(product_ids) == product_embeddings.shape[0]
assert product_embeddings.shape[1] == 384
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-5)

print("Embedding checks passed")

Product IDs: 600
Matrix rows: 600
First 5 norms: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
Embedding checks passed


In [18]:
from collections import Counter

review_counts = dict(
    Counter(review["product_id"] for review in reviews)
)

print("Products with reviews:", len(review_counts))
print("Total reviews:", sum(review_counts.values()))
print("First product reviews:", review_counts[product_ids[0]])

Products with reviews: 600
Total reviews: 6992
First product reviews: 30


In [19]:
import importlib
import support.search

importlib.reload(support.search)

results = support.search.search_products(
    query="A portable waterproof Bluetooth speaker",
    model=model,
    product_embeddings=product_embeddings,
    product_ids=product_ids,
    products=products,
    review_counts=review_counts,
    batch_size=settings.batch_size,
)

for result in results:
    print(
        f"{result['product_id']} | "
        f"{result['similarity']:.4f} | "
        f"{result['review_count']} reviews | "
        f"{result['title']}"
    )

B0B786PFYJ | 0.7942 | 11 reviews | Altec Lansing Mini H2O - Waterproof Bluetooth Speaker, IP67 Certified & Floats in Water, Compact & Portable Speaker for Hiking, Camping, Pool, and Beach
B0BNY1L4JP | 0.7896 | 13 reviews | EDUPLINK Waterproof Portable Bluetooth Speaker - 20W Louder Wireless Speaker with 20 Hours Playtime, TWS Pairing, RGB Lights and TF Slot - Perfect for Beach and Pool (Black)
B09WWG19YS | 0.7580 | 27 reviews | Bose SoundLink Micro: Small Portable Bluetooth Speaker (Waterproof), Midnight Blue
B096GT8CJH | 0.7361 | 17 reviews | JBL Clip 3, Dusty Pink - Waterproof, Durable & Portable Bluetooth Speaker - Up to 10 Hours of Play - Includes Noise-Cancelling Speakerphone & Wireless Streaming
B07VWS6BZ3 | 0.7190 | 8 reviews | JBL Flip 5 Waterproof Portable Wireless Bluetooth Speaker Bundle with divvi! Protective Hardshell Case - Red


In [20]:
for k in (1, 5, 20):
    results = support.search.search_products(
        query="A portable waterproof Bluetooth speaker",
        model=model,
        product_embeddings=product_embeddings,
        product_ids=product_ids,
        products=products,
        review_counts=review_counts,
        batch_size=settings.batch_size,
        top_k=k,
    )

    scores = [result["similarity"] for result in results]

    assert len(results) == k
    assert scores == sorted(scores, reverse=True)
    assert all(result["product_id"] in products for result in results)
    assert all(
        result["title"] == products[result["product_id"]]["title"]
        for result in results
    )
    assert all(
        result["review_count"] == review_counts[result["product_id"]]
        for result in results
    )

    print(f"top_k={k}: all checks passed")

top_k=1: all checks passed
top_k=5: all checks passed
top_k=20: all checks passed


## 4. Consultas de dominio y evidencias

In [21]:
review_query = "Comfortable headphones with good sound"

review_results = support.search.search_products(
    query=review_query,
    model=model,
    product_embeddings=product_embeddings,
    product_ids=product_ids,
    products=products,
    review_counts=review_counts,
    batch_size=settings.batch_size,
    top_k=5,
)

for result in review_results:
    print(
        f"{result['product_id']} | "
        f"{result['similarity']:.4f} | "
        f"{result['review_count']} reviews | "
        f"{result['title']}"
    )

B07CQMZVZ6 | 0.7294 | 30 reviews | Sony MDR7506 Professional Large Diaphragm Headphone
B091K4WYD1 | 0.7166 | 30 reviews | OneOdio Over Ear Headphone, Wired Bass Headsets with 50mm Driver, Foldable Lightweight Headphones with Shareport and Mic for Recording Monitoring Podcast Guitar PC TV - (Red)
B09L3X7KPH | 0.7124 | 7 reviews | LOBKIN Bluetooth Headphone Over Ear, Stereo Wireless Headset with Microphone, Foldable Wireless and Wired Headphones with TF Card MP3 Mode and FM Radio for Smartphones Bluetooth headsets(Blue)
B0B9H3NN8T | 0.7106 | 9 reviews | ARTIX CL650 Wired Headphones with Microphone for Kids & Adults - On-Ear Head Phones with Mic - Foldable Headphone with Wire 3.5mm (Blue)
B08YLDQ45Y | 0.7086 | 19 reviews | CozyPhones Sleep Headphones - Over Ear Headphones from Ultra Thin Cool Mesh Wired for Side Sleepers, Meditation, Running, Laptop, and Phone - Pink Lycra


In [22]:
import json

# Serialización de resultados reales
json_output = json.dumps(review_results)

assert isinstance(json_output, str)
assert len(review_results) == 5

print("JSON serialization: passed")

# Validaciones de la función integrada
invalid_cases = [
    ("", 5),
    ("   ", 5),
    ("headphones", 0),
    ("headphones", 21),
    ("headphones", 2.5),
    ("headphones", True),
]

for query, top_k in invalid_cases:
    try:
        support.search.search_products(
            query=query,
            model=model,
            product_embeddings=product_embeddings,
            product_ids=product_ids,
            products=products,
            review_counts=review_counts,
            batch_size=settings.batch_size,
            top_k=top_k,
        )
        print(f"FAILED: {query!r}, {top_k!r}")
    except ValueError:
        print(f"PASSED: {query!r}, {top_k!r}")

JSON serialization: passed
PASSED: '', 5
PASSED: '   ', 5
PASSED: 'headphones', 0
PASSED: 'headphones', 21
PASSED: 'headphones', 2.5
PASSED: 'headphones', True
